In [161]:
import os
import re
import json
import fitz 
import asyncio
from openai import AsyncOpenAI
from typing import Any
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from types import SimpleNamespace

load_dotenv()

True

In [ ]:
import re as _re

def process_filename(file_path: str | Path, limit_folder=None) -> str:
    path_obj = Path(file_path).resolve()
    return f"\\\\?\\{path_obj}"

def _sanitize_text(text: str) -> str:
    """Remove null bytes and other control characters that break JSON serialization."""
    # Remove null bytes
    text = text.replace("\x00", "")
    # Remove other non-printable control chars except common whitespace (\t \n \r)
    text = _re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    return text

def parse_pdf_text(filepath: str):
    safe_path = process_filename(filepath)
    
    try:
        with open(safe_path, "rb") as f:
            file_bytes = f.read()
            
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        
        pages_list = []
        for page in doc:
            text_content = _sanitize_text(page.get_text("text"))
            page_obj = SimpleNamespace(markdown=text_content)
            pages_list.append(page_obj)
        
        return SimpleNamespace(pages=pages_list)
        
    except OSError as e:
        print(f"Error opening {Path(filepath).name}: {e}")
        return None

def parse_pdf(filepath: str, method: str = "text"):
    if method == "ocr":
        raise NotImplementedError("OCR method is not implemented yet")
    elif method == "text":
        return parse_pdf_text(filepath)
    else:
        raise ValueError("Method must be 'ocr' or 'text'")

In [5]:
filename_to_len = {}
base_dir = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\cleaned_downloads"
pdf_files = glob(os.path.join(base_dir, "*.pdf"))

for filepath in tqdm(pdf_files, desc="Processing PDFs"):
    response = parse_pdf(filepath, method="text")
    if response is not None:
        total_length = sum(len(page.markdown) for page in response.pages)
        filename_to_len[Path(filepath).name] = total_length

Processing PDFs:   0%|          | 0/2456 [00:00<?, ?it/s]

Processing PDFs:  13%|█▎        | 317/2456 [00:35<11:44,  3.03it/s]

MuPDF error: format error: No default Layer config



Processing PDFs:  99%|█████████▊| 2423/2456 [05:19<00:09,  3.31it/s]

MuPDF error: format error: No default Layer config



...

Processing PDFs:  99%|█████████▉| 2428/2456 [05:21<00:08,  3.19it/s]

MuPDF error: format error: No default Layer config



Processing PDFs:  99%|█████████▉| 2429/2456 [05:21<00:08,  3.31it/s]

MuPDF error: format error: No default Layer config



Processing PDFs: 100%|██████████| 2456/2456 [05:26<00:00,  7.52it/s]


In [169]:
TOTAL_TOKEN_USAGE = {
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0,
    "requests": 0,
}
_TOKEN_USAGE_LOCK = asyncio.Lock()


def _usage_val(usage: Any, key: str) -> int:
    if usage is None:
        return 0
    if isinstance(usage, dict):
        value = usage.get(key, 0)
    else:
        value = getattr(usage, key, 0)
    try:
        return int(value or 0)
    except (TypeError, ValueError):
        return 0


async def _add_usage(usage: Any) -> None:
    prompt_tokens = _usage_val(usage, "prompt_tokens")
    completion_tokens = _usage_val(usage, "completion_tokens")
    total_tokens = _usage_val(usage, "total_tokens")
    if total_tokens == 0:
        total_tokens = prompt_tokens + completion_tokens

    async with _TOKEN_USAGE_LOCK:
        TOTAL_TOKEN_USAGE["prompt_tokens"] += prompt_tokens
        TOTAL_TOKEN_USAGE["completion_tokens"] += completion_tokens
        TOTAL_TOKEN_USAGE["total_tokens"] += total_tokens
        TOTAL_TOKEN_USAGE["requests"] += 1


def reset_token_usage() -> None:
    TOTAL_TOKEN_USAGE["prompt_tokens"] = 0
    TOTAL_TOKEN_USAGE["completion_tokens"] = 0
    TOTAL_TOKEN_USAGE["total_tokens"] = 0
    TOTAL_TOKEN_USAGE["requests"] = 0


def get_token_usage_summary() -> dict[str, int]:
    return dict(TOTAL_TOKEN_USAGE)


# Override: keep OpenAI logic unchanged, use vLLM answer-only extraction for vLLM.
async def async_call_llm(client: AsyncOpenAI, model_name: str, prompt: str, max_tokens: int = 1500) -> str:
    messages = [
        {
            "role": "system",
            "content": """
ATURAN KHUSUS:
1. Gunakan Bahasa Indonesia formal yang lugas. 
2. ANTI-BELANDA/LATIN: Jangan gunakan istilah belanda seperti "Verstek", "Niet Ontvankelijke", "Obscuur Libel", "Wanprestasi", "Ex Aequo Et Bono", dsb. 
   - Gunakan: "Putusan tanpa kehadiran", "Gugatan tidak dapat diterima", "Gugatan kabur", "Ingkar janji".
3. ANONIMISASI: Gunakan [Penggugat], [Tergugat], [Objek Sengketa], [Nomor Perkara] dan abstraksi lainnya untuk hal-hal yang bersifat sensitif.
4. FOKUS REGULASI: Prioritaskan interpretasi fakta berdasarkan pasal-pasal di KUHPerdata.
                """
        },
        {"role": "user", "content": prompt},
    ]

    base_url_obj = getattr(client, "base_url", None)
    base_url_str = str(base_url_obj) if base_url_obj is not None else ""
    is_vllm = "localhost:8000" in base_url_str or "127.0.0.1:8000" in base_url_str

    if not is_vllm:
        # Keep OpenAI path as-is for compatibility.
        response = await client.chat.completions.create(
            model=model_name,
            messages=messages,
            max_completion_tokens=max_tokens,
            temperature=0,
        )
        await _add_usage(getattr(response, "usage", None))
        return response.choices[0].message.content

    # vLLM path: use the same reliable extraction behavior as call_vllm_answer_only.
    response = await client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    await _add_usage(getattr(response, "usage", None))

    msg = response.choices[0].message
    content = getattr(msg, "content", None)

    if isinstance(content, str) and content.strip():
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                txt = item.get("text")
            else:
                txt = getattr(item, "text", None)
            if isinstance(txt, str) and txt.strip():
                parts.append(txt.strip())
        if parts:
            return "\n".join(parts)

    return ""


async def process_worker_chunk(
        client: AsyncOpenAI, 
        model_name: str, 
        chunk_id: int, 
        chunk_text: str, 
        blueprint: str, 
        sem: asyncio.Semaphore,
        verbose: bool = True
    ) -> str:
    worker_prompt = f"""
    TUGAS: Anda adalah analis hukum yang sangat teliti. Anda sedang memeriksa BAGIAN #{chunk_id} dari sebuah dokumen putusan.

    Konteks global sebagai panduan:
    {blueprint}

    ATURAN KHUSUS:
    1. OBSERVASI LOKAL: Hanya ekstrak informasi yang BENAR-BENAR MUNCUL dalam teks Chunk #{chunk_id} di bawah ini. Jangan mengulang informasi dari Blueprint jika tidak ada buktinya di teks chunk ini.
    2. IDENTIFIKASI TAHAPAN: Tentukan apakah chunk ini berisi: 'Identitas Para Pihak', 'Duduk Perkara (Kronologi)', 'Pertimbangan Hukum (Ratio Decidendi)', atau 'Amar Putusan'. Fokuskan ekstraksi pada fungsi bagian tersebut.
    3. PENERAPAN GROUND TRUTH: Jika ada pasal yang disebutkan, jangan hanya menyalin isinya. Jelaskan: "Hakim menggunakan pasal ini UNTUK menilai [fakta apa]".

    FORMAT OUTPUT WAJIB:
    ### [A] Analisis Chunk #{chunk_id}
    - **Kategori Dokumen**: (Sebutkan: Identitas/Duduk Perkara/Pertimbangan/Amar)
    - **Detail Unik**: (Ekstrak fakta spesifik atau argumen yang hanya ada di chunk ini)
    - **Logika Hukum**: (Bagaimana pihak atau hakim membangun argumen di bagian ini)

    ### [B] Temuan Regulasi Spesifik (Jika Ada)
    - **Pasal/Doktrin**: [Nomor Pasal]
    - **Konteks di Chunk**: [Mengapa pasal ini muncul di sini?]

    Dilarang mengeluarkan output "Tidak ada data relevan" kecuali chunk tersebut benar-benar kosong atau hanya berisi disclaimer teknis.

    TEKS CHUNK #{chunk_id}:
    {chunk_text}
    """
    async with sem:
        extraction = await async_call_llm(client, model_name, worker_prompt, max_tokens=2500)
        print_line(f"✅ [Worker {chunk_id}] Finished.", verbose=verbose)
        return f"\n--- CHUNK {chunk_id} EXTRACTIONS ---\n{extraction}"

def print_line(text, verbose=True):
    if verbose:
        print(text)

async def run_summarization_pipeline(
        client: AsyncOpenAI, 
        model_name: str, 
        pages: list[str], 
        worker_batch_size: int = 10,
        chunk_size_words: int = 10000,
        overlap_words: int = 200,
        verbose: bool = True
    ):

    print_line(f"🚀 Starting Pipeline using model: {model_name}...", verbose=verbose)
    full_text = " ".join(pages)
    words = full_text.split()
    
    # The Planner
    print_line("\n📝 [Phase 1] Generating Blueprint...", verbose=verbose)
    planner_context = "\n".join(pages[:5] + pages[-5:])
    blueprint_prompt = f"""
    Identifikasi 3 elemen dari dokumen ini:
    1. Subjek & Objek: Siapa [Penggugat], [Tergugat], dan apa [Objek Sengketa]?
    2. Garis Besar Konflik: Inti masalah (misal: jual beli, utang piutang).
    3. Hasil Akhir: Apakah gugatan dikabulkan, ditolak, atau tidak dapat diterima?

    Gunakan informasi ini sebagai konteks untuk proses ekstraksi detail nantinya.
    
    Kembalikan blueprint dalam format poin-poin yang jelas dan terstruktur.
    Document text: {planner_context}
    """
    blueprint = await async_call_llm(client, model_name, blueprint_prompt, max_tokens=1500)
    print_line("✅ Blueprint Generated:\n", verbose=verbose)
    print_line(blueprint, verbose=verbose)

    
    # The Worker
    print_line(f"\n[Phase 2] Spawning Workers (Batch Size: {worker_batch_size})...", verbose=verbose)
    chunks = []
    step = chunk_size_words - overlap_words
    for i in range(0, len(words), step):
        chunk_content = " ".join(words[i : i + chunk_size_words])
        chunks.append(chunk_content)
    
    sem = asyncio.Semaphore(worker_batch_size)
    tasks = [
        asyncio.create_task(process_worker_chunk(client, model_name, idx + 1, chunk_text, blueprint, sem, verbose=verbose))
        for idx, chunk_text in enumerate(chunks)
    ]
    worker_results = await asyncio.gather(*tasks)
    combined_extractions = "\n".join(worker_results)

    # The Synthesizer
    print_line("\n[Phase 3] Synthesizing Final Summary...", verbose=verbose)
    synthesizer_prompt = f"""
Ubah catatan ekstraksi menjadi query yang relevan melalui prosedur berikut.

ATURAN KHUSUS:
1. HAPUS SEMUA ISTILAH BELANDA/LATIN. Jika ada "Verstek", ubah jadi "Putusan tanpa kehadiran". Jika ada "Obscuur Libel", ubah jadi "Gugatan tidak jelas/kabur". Jika ada "NO", ubah jadi "Gugatan tidak dapat diterima".
2. FOKUS KUHPERDATA: Hubungkan temuan fakta dengan pasal-pasal di KUHPerdata
3. BAHASA: jangan gunakan akronim atau istilah yang tidak umum. Gunakan Bahasa Indonesia sebagai abstraksi dari istilah pada notes.
4. Pada bagian Relevansi Pasal, HANYA masukkan pasal yang benar-benar dibahas atau menjadi dasar logika di dalam kalimat query tersebut. Jangan memasukkan semua pasal jika query hanya membahas satu aspek spesifik.
5. Jangan melakukan over-labeling pada Relevansi Pasal. Jika sebuah query membahas tentang kegagalan prosedur atau kejelasan objek, jangan mencantumkan pasal materiil kecuali query tersebut secara eksplisit menanyakan tentang unsur kesalahan atau kerugian.

STRUKTUR OUTPUT:
1. Pertanyaan Hukum Orang Awam: Satu pertanyaan natural dari perspektif non legal. Contoh: "Kenapa saya kalah padahal lawan tidak datang sidang?"
   Relevansi Pasal: [Relevansi: Pasal XXX KUHPerdata, ...].

2. Ringkasan Kasus: Fokus pada Fakta perikatan, tindakan ingkar janji, dan alasan logis hakim menyatakan gugatan kabur karena tuntutan yang saling bertentangan.
   Relevansi Pasal: [Relevansi: Pasal XXX KUHPerdata, ...].

Extracted Notes:
{combined_extractions}
"""

    final_summary = await async_call_llm(client, model_name, synthesizer_prompt, max_tokens=2500)
    print_line("\n================ FINAL SUMMARY ================\n", verbose=verbose)
    print_line(final_summary, verbose=verbose)
    print_line("\n================ TOKEN USAGE ================\n", verbose=verbose)
    print_line(get_token_usage_summary(), verbose=verbose)
    
    return final_summary, combined_extractions

In [170]:
# Token usage helpers (kept in a separate cell for quick inspection/reset).
def reset_token_usage() -> None:
    TOTAL_TOKEN_USAGE["prompt_tokens"] = 0
    TOTAL_TOKEN_USAGE["completion_tokens"] = 0
    TOTAL_TOKEN_USAGE["total_tokens"] = 0
    TOTAL_TOKEN_USAGE["requests"] = 0


def get_token_usage_summary() -> dict[str, int]:
    return dict(TOTAL_TOKEN_USAGE)


print("Current token usage:", get_token_usage_summary())

Current token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'requests': 0}


In [171]:
df = pd.DataFrame(list(filename_to_len.items()), columns=['Filename', 'Length'])

df.sort_values(by='Length', ascending=True, inplace=True)
df.iloc[1800]["Filename"]

'Putusan_PN_DENPASAR_Nomor_1284_Pdt.G_2024_PN_Dps.pdf'

## Inference on Sample

In [ ]:
if __name__ == "__main__":
    ENVIRONMENT = "openai"
    
    if ENVIRONMENT == "openai":
        active_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        active_model = "gpt-5.4-nano-2026-03-17"
    else:
        active_client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
        active_model = "QuantTrio/Qwen3.5-27B-AWQ"

    response = parse_pdf(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\cleaned_downloads\Putusan_PN_DENPASAR_Nomor_1284_Pdt.G_2024_PN_Dps.pdf", method="text")
    pages_list = [page.markdown for page in response.pages]

    reset_token_usage()
    final_output, worker_output = await run_summarization_pipeline(active_client, active_model, pages_list)
    print("\nToken usage summary:", get_token_usage_summary())

🚀 Starting Pipeline using model: gpt-5.4-nano-2026-03-17...

📝 [Phase 1] Generating Blueprint...
✅ Blueprint Generated:
 - **1) Subjek & Objek**
  - **[Penggugat]**: **MARIA LOUISE TUMEWU** (dalam perkara: gugatan perbuatan melawan hukum).
  - **[Tergugat]**: **I KETUT TEKEN**.
  - **[Turut Tergugat]**: **BADAN PERTANAHAN NASIONAL KABUPATEN BADUNG**.
  - **[Objek Sengketa]**: **tanah seluas 800 m²** dengan **Sertifikat Hak Milik No. 5121 Tahun 2001** atas nama **[Penggugat]**, terletak di **Desa Canggu, Kecamatan Kuta, Kabupaten Badung**.  
    - Dalam dalil [Penggugat], persoalan utamanya terkait **bangunan yang menghalangi akses jalan keluar-masuk** menuju tanah tersebut.

- **2) Garis Besar Konflik (inti masalah)**
  - [Penggugat] mendalilkan telah membeli tanah pada **18 Agustus 2001** dan memiliki **SHM No. 5121/2001** seluas **800 m²**.
  - [Penggugat] menyatakan bahwa terdapat **akses jalan/gang** yang seharusnya menjadi fasilitas akses menuju tanahnya (sesuai informasi/tata rua

In [133]:
print(worker_output)


--- CHUNK 1 EXTRACTIONS ---
### [A] Analisis Chunk #1
- **Kategori Dokumen**: **Identitas Para Pihak** dan **Duduk Perkara (Kronologi/Posita Gugatan) serta bagian awal proses persidangan dan Eksepsi/Jawaban Tergugat (Konvensi)**.
- **Detail Unik**:
  - Identitas lengkap para pihak beserta alamatnya, termasuk **[Penggugat] AMRI** dan **[Tergugat] (TERGUGAT I–IV)**.
  - **Posita [Penggugat]** merinci histori perolehan tanah garapan melalui beberapa akta/surat, khususnya:
    - **Akta Pengoperan Hak No. 520** (31 Desember 2016) sebagai dasar perolehan [Penggugat].
    - Obyek sengketa diuraikan seluas **± 3.538 m²** dengan batas-batas tertentu dan dikaitkan dengan **Girik No. 909 Persil 82.S.RR/I** atas nama **MENAK**.
    - [Penggugat] mendalilkan penguasaan/pensertifikatan oleh **TERGUGAT I** melalui **SHM No. 3494/Kamal** dan menilai sertifikat tersebut **tanpa alas hak yang sah** serta mengandung **cacat administrasi/data yuridis-fisik**.
    - [Penggugat] mendalilkan **TERGUGAT I–TE

## Inference on All Files

In [ ]:
from datetime import datetime

async def process_pdf_task(
    sem: asyncio.Semaphore,
    client: AsyncOpenAI,
    model: str,
    idx: int,
    filepath: str,
) -> tuple:
    async with sem:
        response = parse_pdf(os.path.join(base_dir, filepath), method="text")
        if response is None:
            return idx, None, None, 0, 0

        pages_list = [page.markdown for page in response.pages]

        before = get_token_usage_summary()
        final_output, worker_output = await run_summarization_pipeline(
            client, model, pages_list, verbose=False
        )
        after = get_token_usage_summary()

        return (
            idx,
            final_output,
            worker_output,
            after["prompt_tokens"] - before["prompt_tokens"],
            after["completion_tokens"] - before["completion_tokens"],
        )


if __name__ == "__main__":
    ENVIRONMENT = "openai"

    if ENVIRONMENT == "openai":
        active_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        active_model = "gpt-5.4-nano-2026-03-17"
    else:
        active_client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
        active_model = "QuantTrio/Qwen3.5-27B-AWQ"

    sem = asyncio.Semaphore(5)
    indices = df.index[:100].tolist()
    filepaths = df["Filename"].values[:100].tolist()

    tasks = [
        process_pdf_task(sem, active_client, active_model, idx, fp)
        for idx, fp in zip(indices, filepaths)
    ]
    results = await tqdm_asyncio.gather(*tasks, desc="Processing PDFs with Pipeline")

    for idx, final_output, worker_output, input_tokens, output_tokens in results:
        df.at[idx, "processed_at"] = datetime.now()
        df.at[idx, "final_output"] = final_output
        df.at[idx, "worker_output"] = worker_output
        df.at[idx, "input_token_usage"] = input_tokens
        df.at[idx, "output_token_usage"] = output_tokens

In [167]:
os.path.join(base_dir, filepath)

'C:\\Users\\ghana\\OneDrive\\kuliah\\Semester 8\\TA\\experiment\\cleaned_downloads\\Putusan_PN_DENPASAR_Nomor_108_Pdt.P_2019_PN_Dps.pdf'